# Arrays larger than memory

**Optional deep dive, about 10 minutes.** Dask can represent an array larger than memory because it creates or reads chunks only when a result needs them.

We will inspect a large virtual array, then compute only a bounded slice. The notebook does not ask every learner to process hundreds of gigabytes.

In [ ]:
import os
import time
import dask.array as da

test_mode = os.environ.get("PYHPC_TEST_MODE") == "1"
side = 20_000 if test_mode else 200_000
chunk_side = 2_000
virtual = da.random.default_rng(2026).normal(
    0, 1, size=(side, side), chunks=(chunk_side, chunk_side)
)
print("Shape:", virtual.shape)
print("Blocks:", virtual.numblocks)
print(f"Logical size: {virtual.nbytes / 1024**3:.1f} GiB")

Creating `virtual` builds only a task graph. It describes the full array but does not create every array item in memory yet.

In [ ]:
sample = virtual[: 2 * chunk_side, : 2 * chunk_side]
print("Sample blocks:", sample.numblocks)
print(f"Sample size: {sample.nbytes / 1024**2:.1f} MiB")

started = time.perf_counter()
sample_mean = sample.mean().compute(scheduler="threads", num_workers=4)
elapsed = time.perf_counter() - started
print(f"Sample mean: {sample_mean:.6f}")
print(f"Elapsed: {elapsed:.3f} s")

## Think before calculating the full mean

A full `virtual.mean().compute()` would process every item. Dask can limit memory use, but it still has to do all of the calculations. Estimate the bytes, number of tasks, and expected time before calculating the full mean.

In [ ]:
full_task_count = virtual.npartitions
print("Array blocks to process:", full_task_count)
print(f"Logical bytes to generate and reduce: {virtual.nbytes:,}")
print("The full mean is intentionally not calculated in this lesson.")

## Takeaway

Chunks let Dask work with an array larger than memory. Dask still has to read or create the data and perform the calculations. First check the task graph on a small slice that represents the full array.